# ORC Basics — 01: Reading ORC Files

## What you will learn
ORC (Optimized Row Columnar) is a columnar format originally built for Hive.
It is Parquet's main competitor — similar performance but with different internals
and stronger native integration with the Hadoop/Hive ecosystem.

In this notebook:
1. What ORC is and how it differs from Parquet
2. Basic `spark.read.orc()` — file, directory, glob
3. ORC schema inference — why it's faster than CSV
4. Reading with explicit schema
5. ORC metadata — stripes, row index, column statistics
6. ORC vs Parquet — first look at differences


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/orc_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('orc-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

random.seed(42)
N = 200_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2023, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 365))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price,2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset: {N:,} rows")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 05:58:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


26/04/10 05:59:06 WARN TaskSetManager: Stage 0 contains a task of very large size (2548 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 05:59:08 WARN TaskSetManager: Stage 1 contains a task of very large size (2617 KiB). The maximum recommended task size is 1000 KiB.


Dataset: 200,000 rows


## Step 1 — ORC File Layout

ORC stores data in **stripes** (default 64 MB each). Each stripe has:
- Row data (column chunks)
- Row index (every 10,000 rows — for row-level skipping)
- Stripe footer (column statistics: min/max/sum/count)

Unlike Parquet's row groups, ORC stripes have a **3-level index**:
File footer → Stripe → Row index → actual rows.

In [2]:
# Write test ORC
ORC_PATH = f"{DATA_DIR}/orders.orc"
df.write.mode("overwrite").orc(ORC_PATH)

files = glib.glob(f"{ORC_PATH}/*.orc")
size_mb = sum(pathlib.Path(f).stat().st_size for f in files) / 1024/1024
print(f"Written: {N:,} rows | {len(files)} file(s) | {size_mb:.1f} MB")

# Compare with Parquet
PQ_PATH = f"{DATA_DIR}/orders.parquet"
df.write.mode("overwrite").parquet(PQ_PATH)
pq_mb = sum(pathlib.Path(f).stat().st_size for f in glib.glob(f"{PQ_PATH}/*.parquet"))/1024/1024
print(f"Parquet: {pq_mb:.1f} MB  ORC: {size_mb:.1f} MB  ratio: {size_mb/pq_mb:.2f}x")

26/04/10 05:59:09 WARN TaskSetManager: Stage 4 contains a task of very large size (2617 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Written: 200,000 rows | 4 file(s) | 2.5 MB


26/04/10 05:59:10 WARN TaskSetManager: Stage 5 contains a task of very large size (2548 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Parquet: 3.9 MB  ORC: 2.5 MB  ratio: 0.63x


## Step 2 — Basic Read

In [3]:
# Read directory
df_orc = spark.read.orc(ORC_PATH)
print(f"Rows: {df_orc.count():,}")
df_orc.printSchema()
df_orc.show(5)

Rows: 200,000
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)

+--------+-----------+-----------+--------+-------+--------+------+-------+---------+----------+
|order_id|customer_id|    product|category|country|quantity| price|revenue|   status|order_date|
+--------+-----------+-----------+--------+-------+--------+------+-------+---------+----------+
|       1|       4013| Product_58|Clothing|     UK|       2| 29.89|  59.78|cancelled|2023-05-21|
|       2|        489| Product_24|Clothing|     FR|       2|592.54|1185.08|cancelled|2023-01-17|
|       3|       8929|Product_108|Clothing|     BR|      10|  31.4|  314.0|cancelled|2023-04-12|
|     

## Step 3 — Glob and Multiple Paths

In [4]:
# Glob pattern
df_glob = spark.read.orc(f"{ORC_PATH}/part-0000*.orc")
print(f"Glob read: {df_glob.count():,} rows")

# List of paths
files = glib.glob(f"{ORC_PATH}/*.orc")
df_list = spark.read.orc(*files[:2])
print(f"List of 2 files: {df_list.count():,} rows")

# recursiveFileLookup
pathlib.Path(f"{ORC_PATH}/sub").mkdir(exist_ok=True)
df.limit(100).coalesce(1).write.mode("overwrite").orc(f"{ORC_PATH}/sub")
df_rec = spark.read.option("recursiveFileLookup","true").orc(ORC_PATH)
print(f"Recursive: {df_rec.count():,} rows (includes sub/)")

26/04/10 05:59:13 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /workspace/data/orc_basics/orders.orc/part-0000*.orc.
java.io.FileNotFoundException: File /workspace/data/orc_basics/orders.orc/part-0000*.orc does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.s

Glob read: 200,000 rows


IllegalArgumentException: For input string: "/workspace/data/orc_basics/orders.orc/part-00001-3f615a6f-d969-487c-af79-5d49ce7b036e-c000.zstd.orc"

## Step 4 — ORC Metadata via PyArrow

In [ ]:
try:
    import pyarrow.orc as po
    orc_file = glib.glob(f"{ORC_PATH}/*.orc")[0]
    f = po.ORCFile(orc_file)
    print(f"ORC file metadata:")
    print(f"  Rows         : {f.nrows:,}")
    print(f"  Stripes      : {f.nstripes}")
    print(f"  Schema fields: {f.schema.names[:5]}...")
    print(f"  Compression  : {f.compression}")
    if f.nstripes > 0:
        stripe = f.read_stripe(0)
        print(f"  Stripe 0 rows: {stripe.num_rows:,}")
except ImportError:
    print("pyarrow.orc not available — use explain() to see ORC metadata:")
    spark.read.orc(ORC_PATH).filter(F.col("category")=="Electronics").explain(mode="formatted")

## Step 5 — Column Pruning and Predicate Pushdown

In [ ]:
# Column pruning — ORC reads only selected column bytes
t_all, t_one = [], []
for _ in range(3):
    t0=time.time(); spark.read.orc(ORC_PATH).agg(F.sum("revenue")).collect(); t_all.append(time.time()-t0)
    t0=time.time(); spark.read.orc(ORC_PATH).select("revenue").agg(F.sum("revenue")).collect(); t_one.append(time.time()-t0)

print(f"SELECT *       : {sorted(t_all)[1]:.3f}s")
print(f"SELECT revenue : {sorted(t_one)[1]:.3f}s  (column pruning)")
print()

# Predicate pushdown — verify via explain
print("Predicate pushdown:")
spark.read.orc(ORC_PATH).filter(F.col("category")=="Electronics").explain(mode="simple")
print("Look for PushedFilters in OrcScan node")

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 49248)
Traceback (most recent call last):
  File "/usr/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/spark/python/pyspark/accumulators.py", line 299, in handle
    poll(accum_updates)
  File "/opt/spark/python/pyspark/accumulators.py", line 271, in poll
    if self.rfile in r and func():
  File "/opt/spark/python/pyspark/accumulators.py", line 275, in accum_updates
    num_updates = read_int(self.rfile)
  File "/opt/spark/python/pyspark/serializers.py", lin

## Summary

```python
spark.read.orc('/path/')
spark.read.orc('/path/*.orc')          # glob
spark.read.orc('/path1', '/path2')     # multiple paths
spark.read.option('recursiveFileLookup','true').orc('/path/')
```

ORC supports the same column pruning and predicate pushdown as Parquet.
Key difference: ORC uses stripes (default 64 MB) vs Parquet row groups (default 128 MB).